# C1: GRPO-RL для генерации ответов (ROADMAP C)
**Что меняется против параметрического REINFORCE (D2.2):**
- action = СГЕНЕРИРОВАННАЯ ПОСЛЕДОВАТЕЛЬНОСТЬ (токены), не шум параметров;
- advantage = r_j − mean(r группы из G сэмплов) — без EMA-бейзлайна и его
  лага (диагностировано: advantage был всегда положителен);
- length-normalized логпробы, авто-коэффициент по первому событию.

Ожидание: единственная ветка с шансом на положительный вклад RL в
генерации. Пары с MLE D2.1 (shared_init.pt с диска), сиды 42–44.
Время ~30 мин/сид → ~1.5 ч. Resume-safe. Инфраструктура — копии
ячеек experiment.ipynb (+ GRPO-инъекции в тренер и prepare_trainer).

In [1]:
import time, os, gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# CUBLAS_WORKSPACE_CONFIG НЕ включаем: окружение новых сидов должно
# совпадать с завершёнными прогонами (смешивать числовые среды между
# сидами таблицы не хотим); детерминизм уже показан эмпирически
# (каузальный тест, §5.5: бит-в-бит воспроизведение)

# ============================================================
# CELL 1: Imports
# ============================================================
import torch
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    get_cosine_schedule_with_warmup,
    set_seed,
)
from peft import PrefixTuningConfig, TaskType, get_peft_model
from datasets import load_dataset

import numpy as np
import evaluate

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM: {vram_gb:.1f} GB")


<VENV>/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti
VRAM: 15.5 GB


In [2]:
# ============================================================
# CELL D2-1: Конфиг D2 — response generation на ED
# ============================================================
class CFG_D2:
    # ---- Core
    seed = 42
    output_dir = "./gen_dialogue_ed_d2"
    gradient_checkpointing = False
    use_cache = True
    bf16 = True
    fp16 = False

    # ---- Model / PEFT (как D1.6 — проверенная стабильность)
    model_name = "Qwen/Qwen2.5-3B"
    trust_remote_code = True
    peft_kind = "prefix_tuning"
    peft_method = "prefix_tuning"
    num_virtual_tokens = 20
    prefix_projection = True
    init_selection_inits = 4
    init_selection_probe_steps = 200

    # ---- Task
    task_kind = "response_gen"
    dataset_repo = "empathetic_dialogues"
    dataset_revision = "refs/convert/parquet"
    train_size = 6000
    val_size = 200
    test_size = 500
    max_source_len = 128
    max_target_len = 48          # с EOS (генерация — не cls)
    max_total_len = max_source_len + max_target_len
    prompt_template = "Emotion: {emotion}\nSituation: {source}\nResponse:"
    padding_side = "left"
    eval_max_new_tokens = 48
    gen_max_new_tokens = 48
    eval_metrics = ["rouge1", "rouge2", "rougeL", "distinct1", "distinct2"]

    # ---- Schedule (одна фаза, контраст с 1-й эпохи — урок §4.6)
    phase1_epochs = 4
    total_epochs = 4
    batch_size = 8
    eval_batch_size = 8
    gradient_accumulation_steps = 1
    learning_rate = 5e-4
    phase2_lr = 1e-4
    weight_decay = 0.0
    warmup_steps = 200
    lr_scheduler = "cosine"
    max_grad_norm = 1.0
    logging_steps = 10
    eval_strategy = "epoch"
    save_strategy = "epoch"
    save_total_limit = 1
    load_best_model_at_end = False
    remove_unused_columns = False
    dataloader_pin_memory = True
    report_to = "none"

    # ---- Contrast (мягкие значения D1.6: стабильны 15/15)
    alpha = 1.0
    beta = 0.1
    contrast_mode = "offtopic"
    contrast_from_start = True
    contrastive_dropout = 0.1
    contrastive_tau = 0.5
    k_negatives = 1
    contrast_neg_in_graph = False   # D2.3: градиент через негатив (абляция)
    reward_baseline_beta = 0.9      # D2.2b: инерция EMA-бейзлайна REINFORCE

    # ---- RL: в D2.1 ВЫКЛЮЧЕН — сначала ядро темы (контраст);
    # RL вернём в D2.2 (reward = ROUGE-L) после чистого ответа на гипотезу
    gamma = 0.0
    gamma_auto = False
    gamma_target_frac = 0.3
    sigma = 0.02
    rl_interval = 50
    rl_num_batches = 16
    rl_subset_size = 600
    use_reward_baseline = True
    gen_do_sample = False
    gen_temperature = 0.7
    gen_top_p = 0.9
    reward_metric = "rougeL"

    divergence_ce_threshold = 8.0
    guard_start_step = 100
    val_rouge_examples = 200


cfg2 = CFG_D2()
assert cfg2.task_kind == "response_gen"
assert cfg2.gamma == 0.0 and not cfg2.gamma_auto
print("Config D2 OK (response_gen, RL выключен до D2.2)")
print(f"  model: {cfg2.model_name} | src<={cfg2.max_source_len} tgt<={cfg2.max_target_len} (с EOS)")
print(f"  шаблон: {cfg2.prompt_template!r}")
print(f"  contrast: mode={cfg2.contrast_mode}, beta={cfg2.beta}, tau={cfg2.contrastive_tau}")

Config D2 OK (response_gen, RL выключен до D2.2)
  model: Qwen/Qwen2.5-3B | src<=128 tgt<=48 (с EOS)
  шаблон: 'Emotion: {emotion}\nSituation: {source}\nResponse:'
  contrast: mode=offtopic, beta=0.1, tau=0.5


In [3]:
# ============================================================
# CELL D2-2: Данные ED (диалоги) + диалоговые негативы
# Эпизод = (эмоция, ситуация, ПЕРВАЯ реплика по utterance_idx).
# Негативы (детерминированы RNG от имени сплита, не от сида):
#   offtopic   — реплика ДРУГОГО эпизода (неуместность);
#   incoherent — перестановка предложений золотой реплики (связность),
#                фолбэк для коротких — перестановка слов; вырожденные
#                (негатив == позитив) заменяются на offtopic.
# ============================================================
# (фикс: эти импорты и токенизатор в experiment.ipynb живёт в D1-ячейке
# данных, которую мы не копировали — здесь они обязательны)
from huggingface_hub import hf_hub_download, list_repo_files
from datasets import Dataset as HFDataset
if "tok" not in globals():
    tok = AutoTokenizer.from_pretrained(cfg2.model_name, trust_remote_code=True)
    tok.padding_side = cfg2.padding_side
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

import re as _re
import random as _random

SENT_SPLIT = _re.compile(r"(?<=[.!?])\s+")


def load_ed_conversations(split):
    """ED -> [(emotion, situation, первая реплика ДРУГОГО спикера)].

    Особенность ED: utterance_idx 0 — пересказ ситуации её автором
    (тем же спикером), а не ответ. Эмпатичный ответ = первая реплика
    другого speaker_idx (обычно idx 1-2). Без второго спикера 64/17844
    диалогов — пропускаем."""
    files = list_repo_files(cfg2.dataset_repo, repo_type="dataset",
                            revision=cfg2.dataset_revision)
    names = sorted(f for f in files
                   if f.startswith(f"default/{split}/") and f.endswith(".parquet"))
    assert names, f"нет parquet-шардов для split={split}"
    paths = [hf_hub_download(cfg2.dataset_repo, n, repo_type="dataset",
                             revision=cfg2.dataset_revision) for n in names]
    ds = load_dataset("parquet", data_files=paths, split="train")
    by_conv = {}
    for r in ds:
        by_conv.setdefault(r["conv_id"], []).append(r)
    convs = []
    for cid in sorted(by_conv):
        rows = sorted(by_conv[cid], key=lambda r: r["utterance_idx"])
        spk0 = rows[0]["speaker_idx"]
        reply = next((r for r in rows if r["speaker_idx"] != spk0), None)
        if reply is None:
            continue
        utt = reply["utterance"].replace("_comma_", ",").strip()
        sit = rows[0]["prompt"].strip()
        emo = rows[0]["context"].strip()
        if utt and sit and emo:
            convs.append({"emotion": emo, "situation": sit, "response": utt})
    return convs


def make_incoherent(text, rng):
    """Негатив-перестановка: предложения (2 = детерминированный свап,
    >2 = shuffle с retry); для 1-предложечных — слова (от 2 слов, с
    retry); None = вырожден (одно слово/одно предложение из 1 слова)."""
    sents = [s for s in SENT_SPLIT.split(text.strip()) if s]
    if len(sents) == 2:
        return f"{sents[1]} {sents[0]}".strip()
    if len(sents) > 2:
        for _ in range(3):
            rng.shuffle(sents)
            cand = " ".join(sents).strip()
            if cand != text.strip():
                return cand
    words = text.split()
    if len(words) >= 2:
        for _ in range(3):
            rng.shuffle(words)
            cand = " ".join(words)
            if cand != text.strip():
                return cand
    return None


def attach_negatives(rows, split, k_off=4):
    """neg_off (СОВМЕСТИМО с D2.1: j=perm[i], j!=i), neg_off2..k_off —
    дополнительные различные чужие ответы (D2.3: k негативов для
    Σ_j из Eq. 3), neg_inc — перестановка."""
    rng_off = _random.Random(f"offtopic-{split}")
    perm = list(range(len(rows)))
    rng_off.shuffle(perm)
    rng_inc = _random.Random(f"incoherent-{split}")
    n_degen = 0
    n = len(rows)
    for i, c in enumerate(rows):
        used = {i}
        j = perm[i]
        if j == i:
            j = (j + 1) % n
        used.add(j)
        c["neg_off"] = rows[j]["response"]
        extra, step = 0, 1
        while extra < k_off - 1 and step < n:
            cand = perm[(i + step) % n]
            if cand == i:
                cand = (cand + 1) % n
            if cand not in used:
                used.add(cand)
                c[f"neg_off{extra + 2}"] = rows[cand]["response"]
                extra += 1
            step += 1
        while extra < k_off - 1:      # датасет мал — дублируем (не бывает)
            c[f"neg_off{extra + 2}"] = c["neg_off"]
            extra += 1
        inc = make_incoherent(c["response"], rng_inc)
        if inc is None or inc.strip() == c["response"].strip():
            inc = c["neg_off"]      # вырожденный негатив -> offtopic
            n_degen += 1
        c["neg_inc"] = inc
    return n_degen


def prepare_split(convs_list, name):
    rows = [dict(r) for r in convs_list]
    ndeg = attach_negatives(rows, name)
    return rows, ndeg


ed2_train_full = load_ed_conversations("train")
ed2_val_full = load_ed_conversations("validation")
ed2_test_full = load_ed_conversations("test")
print(f"ED dialogues: train {len(ed2_train_full)} | "
      f"val {len(ed2_val_full)} | test {len(ed2_test_full)}")
assert cfg2.rl_subset_size + cfg2.train_size <= len(ed2_train_full)

rows_rl,    deg_rl    = prepare_split(ed2_train_full[:cfg2.rl_subset_size], "rl")
rows_train, deg_train = prepare_split(
    ed2_train_full[cfg2.rl_subset_size:cfg2.rl_subset_size + cfg2.train_size], "train")
rows_val,   deg_val   = prepare_split(ed2_val_full[:cfg2.val_size], "val")
rows_test,  deg_test  = prepare_split(ed2_test_full[:cfg2.test_size], "test")
print(f"Вырожденных (incoherent->offtopic) негативов: "
      f"train {deg_train}/{len(rows_train)}, val {deg_val}, test {deg_test}")

raw2_rl    = HFDataset.from_list(rows_rl)
raw2_train = HFDataset.from_list(rows_train)
raw2_val   = HFDataset.from_list(rows_val)
raw2_test  = HFDataset.from_list(rows_test)


NEG_FIELD_TO_PFX = {"neg_off": "negoff", "neg_off2": "negoff2",
                    "neg_off3": "negoff3", "neg_off4": "negoff4",
                    "neg_inc": "neginc"}


def tokenize_d2(examples):
    neg_fields = [f for f in NEG_FIELD_TO_PFX if f in examples]
    pfxs = [""] + [NEG_FIELD_TO_PFX[f] for f in neg_fields]
    # пустой префикс -> "input_ids"; негативный -> "negoff_input_ids"
    # (баг было: f"{p}{k}" без "_" для негативных префиксов)
    out = {}
    for p in pfxs:
        for k in ("input_ids", "attention_mask", "labels"):
            out[k if p == "" else f"{p}_{k}"] = []
    max_total = cfg2.max_total_len

    def enc_target(text):
        ids = tok(" " + text, add_special_tokens=False, truncation=True,
                  max_length=cfg2.max_target_len - 1)["input_ids"]
        return ids + [tok.eos_token_id]

    base = zip(examples["emotion"], examples["situation"], examples["response"])
    for bi, (emo, sit, resp) in enumerate(base):
        src = cfg2.prompt_template.format(emotion=emo, source=sit)
        src_ids = tok(src, add_special_tokens=False, truncation=True,
                      max_length=cfg2.max_source_len)["input_ids"]
        if not src_ids:
            src_ids = [tok.pad_token_id]

        tgt_ids = enc_target(resp)
        if len(src_ids) + len(tgt_ids) > max_total:
            src_ids = src_ids[:max(1, max_total - len(tgt_ids))]
        out["input_ids"].append(src_ids + tgt_ids)
        out["attention_mask"].append([1] * (len(src_ids) + len(tgt_ids)))
        out["labels"].append([-100] * len(src_ids) + tgt_ids)

        for f in neg_fields:
            pfx = NEG_FIELD_TO_PFX[f]
            n_ids = enc_target(examples[f][bi])
            if len(src_ids) + len(n_ids) > max_total:
                n_ids = n_ids[:max(1, max_total - len(src_ids))]
            out[f"{pfx}_input_ids"].append(src_ids + n_ids)
            out[f"{pfx}_attention_mask"].append([1] * (len(src_ids) + len(n_ids)))
            out[f"{pfx}_labels"].append([-100] * len(src_ids) + n_ids)
    return out


train2_ds = raw2_train.map(tokenize_d2, batched=True, remove_columns=raw2_train.column_names)
val2_ds   = raw2_val.map(tokenize_d2, batched=True, remove_columns=raw2_val.column_names)
test2_ds  = raw2_test.map(tokenize_d2, batched=True, remove_columns=raw2_test.column_names)
rl2_ds    = raw2_rl.map(tokenize_d2, batched=True, remove_columns=raw2_rl.column_names)
rl2_ds = rl2_ds.add_column("ref_idx", list(range(len(rl2_ds))))

rl2_references   = raw2_rl["response"]
val2_references  = raw2_val["response"]
test2_references = raw2_test["response"]

for d in (train2_ds, val2_ds, test2_ds, rl2_ds):
    d.set_format("torch")

# ВАЖНО: перезапривязываем глобальные имена датасетов к D2 —
# run_experiment / prepare_trainer / probe работают с ними.
# D1-агрегация читает только JSON с диска, конфликта нет.
train_ds, val_ds, test_ds, rl_ds = train2_ds, val2_ds, test2_ds, rl2_ds
rl_references, val_references, test_references = (
    rl2_references, val2_references, test2_references)

print(f"D2: train {len(train2_ds)} | val {len(val2_ds)} | "
      f"test {len(test2_ds)} | RL hold-out {len(rl2_ds)} (для D2.2)")
assert (train2_ds[0]["labels"] != -100).sum() > 0
assert train2_ds[0]["labels"][-1].item() == tok.eos_token_id, \
    "EOS обязан быть последним токеном таргета (генерация)"
print("Инварианты OK: супервизируемые токены есть, EOS в конце")

ED dialogues: train 17780 | val 2758 | test 2540
Вырожденных (incoherent->offtopic) негативов: train 41/6000, val 0, test 1


Map: 100%|██████████| 600/600 [00:00<00:00, 2040.51 examples/s]

D2: train 6000 | val 200 | test 500 | RL hold-out 600 (для D2.2)
Инварианты OK: супервизируемые токены есть, EOS в конце


In [4]:
# ============================================================
# CELL D2-4: метрики генерации — Distinct-n (+ BERTScore опционально)
# ============================================================
def distinct_n(texts):
    """Distinct-1/2: доля уникальных уни-/биграмм по корпусу генераций."""
    unis, bis = set(), set()
    ntok, npairs = 0, 0
    for t in texts:
        w = t.split()
        unis.update(w)
        bis.update(zip(w, w[1:]))
        ntok += len(w)
        npairs += max(len(w) - 1, 0)
    return len(unis) / max(ntok, 1), len(bis) / max(npairs, 1)

try:
    from bert_score import score as _bert_score_fn
    HAVE_BERTSCORE = True
    print("bert-score установлен: BERTScore можно считать по predictions")
except Exception:
    HAVE_BERTSCORE = False
    print("bert-score НЕ установлен (опционально: pip install bert-score); "
          "BERTScore считается отдельной ячейкой по сохранённым predictions")
print("distinct_n готов")

bert-score установлен: BERTScore можно считать по predictions
distinct_n готов


In [5]:
# ============================================================
# CELL 5: Reward function
# D1 (emotion_cls): accuracy матчинга сгенерированного слова к 32 меткам
# summarization (наследие): ROUGE-L против референса
# ============================================================
rouge_metric = evaluate.load("rouge")

@torch.no_grad()
def compute_generation_reward(model, input_ids, attention_mask, references, tokenizer,
                              task_cfg=None):
    """Reward на ДАННОМ примере против его референса. Greedy ->
    детерминированный reward (аналог accuracy в статье).

    Для task_kind="emotion_cls": references = строки-эмоции,
    reward = доля примеров, где match_emotion(генерация) == метка.
    Для "summarization": reward = ROUGE-L.
    """
    # D2.2: task_cfg — конфиг ЗАДАЧИ (глобальный cfg может быть чужим,
    # например D1-классификацией); по умолчанию прежнее поведение
    c = task_cfg if task_cfg is not None else cfg
    model.eval()

    old_use_cache = getattr(model.config, "use_cache", True)
    model.config.use_cache = True

    try:
        gen_kwargs = dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=c.gen_max_new_tokens,
            do_sample=c.gen_do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if c.gen_do_sample:
            gen_kwargs.update(temperature=c.gen_temperature, top_p=c.gen_top_p)

        gen_ids = model.generate(**gen_kwargs)
    finally:
        model.config.use_cache = old_use_cache

    new_tokens = gen_ids[:, input_ids.shape[1]:]
    generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    generated_texts = [x.strip() if x.strip() else " " for x in generated_texts]

    assert len(generated_texts) == len(references), \
        f"predictions/references mismatch: {len(generated_texts)} vs {len(references)}"

    if c.task_kind == "emotion_cls":
        reward = float(np.mean([
            match_emotion(g) == r
            for g, r in zip(generated_texts, references)
        ]))
    else:
        results = rouge_metric.compute(
            predictions=generated_texts,
            references=references,
            rouge_types=["rougeL"],
        )
        reward = float(results["rougeL"])

    model.train()
    return reward, generated_texts

print("Reward function defined: accuracy (emotion_cls) / ROUGE-L (summarization).")


Reward function defined: accuracy (emotion_cls) / ROUGE-L (summarization).


In [6]:
# ============================================================
# CELL 6: TwoPhaseTrainerGen — адаптация Algorithm 1
# D1.5: contrast_mode = "mask" | "wrong_label"; contrast_from_start
# D1.6: RL reward на ЧИСТОМ входе (strip золотой метки — баг эха)
# ============================================================
class TwoPhaseTrainerGen(Trainer):
    def __init__(
        self,
        model=None,
        args=None,
        phase1_epochs: int = 3,
        alpha: float = 1.0,
        beta: float = 0.1,
        gamma: float = 2e-5,
        gamma_auto: bool = True,
        gamma_target_frac: float = 0.3,
        sigma: float = 0.02,
        contrast_mode: str = "mask",
        contrast_from_start: bool = False,
        contrastive_dropout: float = 0.1,
        contrastive_tau: float = 0.5,
        k_negatives: int = 1,
        rl_interval: int = 50,
        rl_num_batches: int = 16,
        rl_subset_size: int = 200,
        use_reward_baseline: bool = True,
        phase2_lr: float = 2e-4,
        divergence_ce_threshold: float = 6.0,
        guard_start_step: int = 200,
        rl_dataset=None,
        rl_references=None,
        tokenizer=None,
        task_cfg=None,
        contrast_neg_in_graph: bool = False,
        reward_baseline_beta: float = 0.9,
        rl_algo: str = "reinforce",        # "reinforce" | "grpo"
        grpo_batch: int = 2,               # контекстов на событие
        grpo_G: int = 4,                   # сэмплов на контекст
        grpo_temperature: float = 0.7,
        grpo_top_p: float = 0.9,
        grpo_use_std: bool = False,        # Dr.GRPO: без std-нормализации
        **kwargs
    ):
        self.phase1_epochs = phase1_epochs
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.gamma_auto = gamma_auto
        self.gamma_target_frac = gamma_target_frac
        self.sigma = sigma
        self.contrast_mode = contrast_mode
        self.contrast_from_start = contrast_from_start
        self.contrastive_dropout = contrastive_dropout
        self.contrastive_tau = contrastive_tau
        self.k_negatives = k_negatives
        self.rl_interval = rl_interval
        self.rl_num_batches = rl_num_batches
        self.rl_subset_size = rl_subset_size
        self.use_reward_baseline = use_reward_baseline
        self.phase2_lr = phase2_lr
        self.divergence_ce_threshold = divergence_ce_threshold
        self.guard_start_step = guard_start_step
        self.rl_dataset = rl_dataset
        self.rl_references = rl_references
        self.tokenizer = tokenizer
        self.task_cfg = task_cfg          # D2.2: конфиг задачи для reward
        self.rl_reward_history = []       # D2.2: трекинг наград по шагам
        self.contrast_neg_in_graph = contrast_neg_in_graph  # D2.3
        self.reward_baseline_beta = reward_baseline_beta    # D2.2b
        self.rl_algo = rl_algo                              # C1: GRPO
        self.grpo_batch = grpo_batch
        self.grpo_G = grpo_G
        self.grpo_temperature = grpo_temperature
        self.grpo_top_p = grpo_top_p
        self.grpo_use_std = grpo_use_std

        # Флаги состояний
        self._phase2_lr_reset_done = False
        self._gamma_calibrated = False
        self._last_rl_step = -1
        self._last_log_step = -1
        self._reward_baseline = 0.0
        self._ce_ema = None
        self._divergence_reported = False

        super().__init__(model=model, args=args, **kwargs)

        # Параметры префикса собираем ПОСЛЕ super().__init__
        self.prefix_params = [
            (n, p) for n, p in self.model.named_parameters()
            if "prompt_encoder" in n and p.requires_grad
        ]
        print(f"[Trainer] Prefix params collected: {len(self.prefix_params)}")

        # Цели контраста (для mode="mask") — ТОЛЬКО embedding-слой
        self.contrast_params = [
            (n, p) for n, p in self.prefix_params if "embedding" in n
        ] or self.prefix_params
        print(f"[Trainer] Contrast: mode={self.contrast_mode}, "
              f"from_start={self.contrast_from_start}, beta={self.beta}")

        if self.rl_dataset is not None:
            self._rl_loader = DataLoader(
                self.rl_dataset,
                batch_size=self.args.per_device_train_batch_size,
                collate_fn=self.data_collator,
            )
            self._rl_iter = iter(self._rl_loader)

    # ----------------------------------------------------------
    # Утилиты
    # ----------------------------------------------------------
    @staticmethod
    def _model_inputs(inputs):
        """Только те ключи, которые понимает model.forward. Коллатор
        дополнительно кладёт neg_*/ref_idx — их в модель нельзя."""
        return {k: v for k, v in inputs.items()
                if k in ("input_ids", "attention_mask", "labels")}

    def _next_rl_batch(self):
        try:
            return next(self._rl_iter)
        except StopIteration:
            self._rl_iter = iter(self._rl_loader)
            return next(self._rl_iter)

    def _restore_params(self, params, snapshot):
        with torch.no_grad():
            for n, p in params:
                p.data.copy_(snapshot[n])

    def _custom_scale(self):
        """ЕДИНЫЙ множитель для ВСЕГО лосса, включая CE (урок ит. 2
        исследования-1: иначе эффективный LR x gradient_accumulation)."""
        trainer_divides = not getattr(self, "model_accepts_loss_kwargs", False)
        return 1.0 if trainer_divides else 1.0 / self.args.gradient_accumulation_steps

    def _reset_lr_for_phase2(self):
        if self.optimizer is None or self.lr_scheduler is None:
            return
        remaining = max(1, self.state.max_steps - self.state.global_step)
        for pg in self.optimizer.param_groups:
            pg["lr"] = self.phase2_lr
            pg["initial_lr"] = self.phase2_lr
        self.lr_scheduler = get_cosine_schedule_with_warmup(
            self.optimizer, num_warmup_steps=0, num_training_steps=remaining,
        )
        print(f"\n[Phase 2] LR reset to {self.phase2_lr}, "
              f"new cosine over remaining {remaining} steps\n")

    def _pool_hidden(self, hidden, labels):
        """Пуллинг по токенам таргета (labels != -100): для позитива —
        токены верной метки, для негатива — токены чужой метки."""
        if hidden.size(1) != labels.size(1):
            hidden = hidden[:, -labels.size(1):, :]
        tgt_mask = (labels != -100).unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * tgt_mask).sum(1) / tgt_mask.sum(1).clamp(min=1)
        return F.normalize(pooled.float(), dim=-1)

    # ----------------------------------------------------------
    # Contrastive loss — два режима негативов
    # ----------------------------------------------------------
    def _contrastive_loss(self, model, inputs, h_pos):
        """L = log1 + sum_j exp((sim(h_i, h_j^-) - 1) / tau))

        mode="mask" (рецепт статьи): h_j^- — представление ТОГО ЖЕ входа
        под замаскированным embedding-слоем промпта. Позитив тривиален
        (sim(h,h)=1 — вырожденность Eq. 3, см. FINAL_ANALYSIS §4).

        mode="wrong_label" (ядро темы): негатив — (ситуация, ЖЁСТКАЯ
        чужая эмоция из группы путаемости) — реальная конкурирующая
        альтернатива."""
        if self.k_negatives <= 0:
            return h_pos.new_zeros(())

        h_negs = []

        # D2: негативы из neg-колонок, собранных при токенизации
        # (neg = wrong-label для cls; negoff*/neginc для response_gen).
        # D2.3: offtopic4 = k=4 негативов (Σ_j как в Eq.3 статьи);
        # oftopic_grad = градиент и через негатив (симметричный апдейт)
        COLUMN_MODES = {
            "wrong_label": ["neg"],
            "offtopic": ["negoff"],
            "offtopic_grad": ["negoff"],
            "offtopic4": ["negoff", "negoff2", "negoff3", "negoff4"],
            "incoherent": ["neginc"],
        }
        if self.contrast_mode in COLUMN_MODES:
            for pfx in COLUMN_MODES[self.contrast_mode]:
                neg_fwd = {
                    "input_ids": inputs[f"{pfx}_input_ids"],
                    "attention_mask": inputs[f"{pfx}_attention_mask"],
                }
                if getattr(self, "contrast_neg_in_graph", False):
                    out_neg = model(**neg_fwd, output_hidden_states=True)
                else:
                    with torch.no_grad():
                        out_neg = model(**neg_fwd, output_hidden_states=True)
                h_negs.append(self._pool_hidden(
                    out_neg.hidden_states[-1], inputs[f"{pfx}_labels"]))
                del out_neg
        else:
            neg_inputs = {k: v for k, v in self._model_inputs(inputs).items()
                          if k != "labels"}
            labels = inputs["labels"]
            original_data = {n: p.data.clone() for n, p in self.contrast_params}
            try:
                for _ in range(self.k_negatives):
                    for n, p in self.contrast_params:
                        mask = (torch.rand_like(p.float()) > self.contrastive_dropout).to(p.dtype)
                        p.data.copy_((original_data[n] * mask).to(p.dtype))
                    with torch.no_grad():
                        out_neg = model(**neg_inputs, output_hidden_states=True)
                    h_neg = self._pool_hidden(out_neg.hidden_states[-1], labels)
                    h_negs.append(h_neg)
                    del out_neg
                    self._restore_params(self.contrast_params, original_data)
            finally:
                self._restore_params(self.contrast_params, original_data)

        h_negs = torch.stack(h_negs, dim=1)  # (B, K, D)

        tau = self.contrastive_tau
        neg_scores = torch.einsum("bd,bkd->bk", h_pos, h_negs) / tau
        loss = torch.log1p(torch.exp(neg_scores - 1.0 / tau).sum(dim=1)).mean()
        return loss

    # ----------------------------------------------------------
    # RL loss (REINFORCE, Eq. 4 статьи)
    # ----------------------------------------------------------
    def _rl_loss(self, model):
        log_probs = []
        noisy_values = {}
        for n, p in self.prefix_params:
            p32 = p.float()
            eps = torch.randn_like(p32) * self.sigma
            noisy = (p32.detach() + eps).detach()
            log_probs.append(
                Normal(loc=p32, scale=self.sigma).log_prob(noisy).sum()
            )
            noisy_values[n] = noisy

        original_data = {n: p.data.clone() for n, p in self.prefix_params}
        rewards = []
        last_texts, last_refs = None, None
        try:
            for n, p in self.prefix_params:
                p.data.copy_(noisy_values[n].to(p.dtype))

            for _ in range(self.rl_num_batches):
                rl_batch = self._next_rl_batch()
                idx = rl_batch.pop("ref_idx", None)
                if idx is None:
                    raise RuntimeError(
                        "ref_idx потерян коллатором: causal_lm_collator обязан "
                        "пробрасывать ref_idx"
                    )
                # D1.6: ЧИСТЫЙ reward — отрезаем золотую метку до generate
                # (баг эха: иначе reward мерил повторение подсказанной метки)
                src_ids, src_mask = strip_label_tokens(rl_batch)
                batch = {
                    "input_ids": src_ids.to(model.device),
                    "attention_mask": src_mask.to(model.device),
                }
                refs = [self.rl_references[i] for i in idx.tolist()]
                r, texts = compute_generation_reward(
                    model, batch["input_ids"], batch["attention_mask"],
                    refs, self.tokenizer, task_cfg=self.task_cfg,
                )
                rewards.append(r)
                last_texts, last_refs = texts, refs
            reward = float(np.mean(rewards))
            self.rl_reward_history.append(
                {"step": int(self.state.global_step), "reward": reward})
        finally:
            self._restore_params(self.prefix_params, original_data)

        if self.use_reward_baseline:
            advantage = reward - self._reward_baseline
            _beta_b = self.reward_baseline_beta   # D2.2b: инерция EMA
            self._reward_baseline = (
                _beta_b * self._reward_baseline + (1 - _beta_b) * reward)
        else:
            advantage = reward

        raw_rl = -advantage * torch.stack(log_probs).sum()

        if self.gamma_auto and not self._gamma_calibrated:
            raw_abs = abs(float(raw_rl.detach()))
            if raw_abs < 1e-6 or self._ce_ema is None:
                print("[RL] gamma calibration postponed")
            else:
                accum = self.args.gradient_accumulation_steps
                self.gamma = float(np.clip(
                    self.gamma_target_frac * self._ce_ema * accum / raw_abs,
                    1e-8, 1.0,
                ))
                self._gamma_calibrated = True
                print(f"[RL] gamma auto-calibrated to {self.gamma:.3e}")

        loss_rl = self.gamma * raw_rl

        if last_texts is not None:
            print(f"[RL sample] REF: {last_refs[0][:110]!r}")
            print(f"[RL sample] GEN: {last_texts[0][:110]!r}")
        print(f"[RL step {self.state.global_step}] reward={reward:.4f} "
              f"adv={advantage:+.4f} gamma={self.gamma:.2e}")
        return loss_rl, reward

    # ----------------------------------------------------------
    # C1: GRPO-RL — групповой REINFORCE по ТОКЕНАМ генераций
    # (action = последовательность; advantage = r_j − mean(r группы),
    # что убирает baseline-лаг, диагностированный в параметрическом
    # REINFORCE). Длина-нормализованные логпробы; коэффициент —
    # авто-калибровка по первому событию (как у γ).
    # ----------------------------------------------------------
    def _rouge_per_example(self, texts, refs):
        res = rouge_metric.compute(predictions=texts, references=refs,
                                   rouge_types=["rougeL"],
                                   use_aggregator=False)["rougeL"]
        return [float(x) for x in res]

    def _grpo_loss(self, model):
        B = self.grpo_batch
        G = self.grpo_G
        batch = self._next_rl_batch()
        idx = batch.pop("ref_idx", None)
        if idx is None:
            raise RuntimeError("ref_idx потерян коллатором")
        src_ids, src_mask = strip_label_tokens(batch)
        src_ids = src_ids[:B].to(model.device)
        src_mask = src_mask[:B].to(model.device)
        refs = [self.rl_references[i] for i in idx.tolist()[:B]]
        Bn = src_ids.size(0)

        # 1) сэмплируем G ответов на каждый контекст
        exp_ids = src_ids.repeat_interleave(G, dim=0)
        exp_mask = src_mask.repeat_interleave(G, dim=0)
        model.eval()
        old_cache = getattr(model.config, "use_cache", True)
        model.config.use_cache = True
        try:
            with torch.no_grad():
                gen = model.generate(
                    input_ids=exp_ids, attention_mask=exp_mask,
                    max_new_tokens=self.task_cfg.gen_max_new_tokens,
                    do_sample=True,
                    temperature=self.grpo_temperature, top_p=self.grpo_top_p,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id)
        finally:
            model.config.use_cache = old_cache
            model.train()
        new_tok = gen[:, exp_ids.size(1):]          # (Bn*G, T)
        texts = self.tokenizer.batch_decode(new_tok, skip_special_tokens=True)

        # 2) награды и групповые advantage (mean-centering)
        rep_refs = [refs[i // G] for i in range(Bn * G)]
        r = np.array(self._rouge_per_example(texts, rep_refs), dtype=np.float32)
        adv = torch.zeros(Bn * G, device=model.device)
        gstd = []
        for gi in range(Bn):
            grp = r[gi * G:(gi + 1) * G]
            a = grp - grp.mean()
            gstd.append(float(grp.std()))
            if self.grpo_use_std and grp.std() > 1e-4:
                a = a / (grp.std() + 1e-4)
            adv[gi * G:(gi + 1) * G] = torch.from_numpy(a).to(model.device)
        adv = adv.detach()

        # 3) teacher-forced логпробы сгенерированных токенов
        eos_id = self.tokenizer.eos_token_id
        has_eos = (new_tok == eos_id).any(dim=1)
        ends = (new_tok == eos_id).float().argmax(dim=1)
        cut = torch.where(has_eos, ends + 1,
                          torch.full_like(ends, new_tok.size(1)))
        full_ids = torch.cat([exp_ids, new_tok], dim=1)
        full_mask = torch.cat([exp_mask, torch.ones_like(new_tok)], dim=1)
        # ЧАНКИ: полный vocab-логит на всех Bn*G строках сразу = ~2 ГБ
        # поверх графа CE -> OOM. Чанк из 8 строк + fused cross_entropy
        # (без материализации log_softmax по словарю).
        tok_lp_rows = []
        CHUNK = 8
        for i in range(0, full_ids.size(0), CHUNK):
            fi = full_ids[i:i + CHUNK]
            fm = full_mask[i:i + CHUNK]
            lg = model(input_ids=fi, attention_mask=fm).logits[:, :-1, :]
            neg_lp = F.cross_entropy(
                lg.reshape(-1, lg.size(-1)), fi[:, 1:].reshape(-1),
                reduction="none").view(fi.size(0), -1)
            tok_lp_rows.append(-neg_lp)
            del lg, neg_lp
        tok_lp = torch.cat(tok_lp_rows, dim=0)
        new_len = new_tok.size(1)
        pos = torch.arange(new_len, device=model.device).unsqueeze(0)
        m = pos < cut.unsqueeze(1)
        m_full = torch.zeros_like(tok_lp, dtype=torch.bool)
        m_full[:, -new_len:] = m
        denom = m_full.sum(dim=1).clamp(min=1)
        seq_lp = (tok_lp * m_full).sum(dim=1) / denom   # length-normalized
        raw = -(adv * seq_lp).mean()

        # 4) авто-калибровка коэффициента (та же схема, что γ)
        if self.gamma_auto and not self._gamma_calibrated:
            raw_abs = abs(float(raw.detach()))
            if raw_abs < 1e-6 or self._ce_ema is None:
                print("[GRPO] калибровка коэффициента отложена")
            else:
                self.gamma = float(np.clip(
                    self.gamma_target_frac * self._ce_ema / raw_abs, 1e-8, 1.0))
                self._gamma_calibrated = True
                print(f"[GRPO] коэффициент откалиброван: {self.gamma:.4f}")
        loss = self.gamma * raw

        reward = float(r.mean())
        self.rl_reward_history.append(
            {"step": int(self.state.global_step), "reward": reward,
             "group_std": float(np.mean(gstd))})
        if self.state.global_step % self.rl_interval == 0:
            print(f"[GRPO step {self.state.global_step}] reward={reward:.4f} "
                  f"group_std={np.mean(gstd):.4f} coef={self.gamma:.4f}")
            print(f"[GRPO sample] REF: {refs[0][:90]!r}")
            print(f"[GRPO sample] GEN: {texts[0][:90]!r} (r={r[0]:.3f}) | "
                  f"GEN: {texts[1][:90]!r} (r={r[1]:.3f})")
        return loss, reward

    # ----------------------------------------------------------
    # Основной лосс
    # ----------------------------------------------------------
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        current_epoch = int(self.state.epoch) if self.state.epoch else 0
        in_phase2 = current_epoch >= self.phase1_epochs

        if in_phase2 and not self._phase2_lr_reset_done:
            self._phase2_lr_reset_done = True
            self._reset_lr_for_phase2()

        use_contrast = self.beta != 0.0 and (in_phase2 or self.contrast_from_start)

        # C1-память: GRPO-событие ЗАМЕЩАЕТ CE-шаг (шаг из rl_interval).
        # Причина: teacher-forced GRPO-граф поверх живого CE-графа того
        # же шага = два стека активаций -> OOM на 15.5 ГБ при любом
        # grpo_batch. Замещение держит один граф; 30 шагов из 3000 CE
        # пропускаются (1%) — на балансе лоссов не сказывается.
        _rl_on = ((self.gamma != 0.0 or self.gamma_auto)
                  and self.rl_dataset is not None)
        if (model.training
                and getattr(self, "rl_algo", "reinforce") == "grpo"
                and _rl_on
                and self.state.global_step % self.rl_interval == 0
                and self._last_rl_step != self.state.global_step):
            self._last_rl_step = self.state.global_step
            loss_rl, reward_val = self._grpo_loss(model)
            _loss = self._custom_scale() * loss_rl
            if (self.state.global_step % 10 == 0
                    and self._last_log_step != self.state.global_step):
                self._last_log_step = self.state.global_step
                print(f"[Step {self.state.global_step}] GRPO-шаг (CE замещён) | "
                      f"RL: {_loss.item():.3f} | Reward: {reward_val:.4f}")
            return (_loss, None) if return_outputs else _loss

        model_inputs = self._model_inputs(inputs)
        outputs = model(**model_inputs, output_hidden_states=use_contrast)
        loss_mle = outputs.loss

        ce_now = loss_mle.item()
        self._ce_ema = ce_now if self._ce_ema is None else 0.95 * self._ce_ema + 0.05 * ce_now

        if (self.state.global_step >= self.guard_start_step
                and self._ce_ema > self.divergence_ce_threshold
                and not self._divergence_reported):
            self._divergence_reported = True
            print(f"\n!!! CE/token EMA = {self._ce_ema:.2f} > "
                  f"{self.divergence_ce_threshold} — расходимость. "
                  f"Останавливаю обучение (шаг {self.state.global_step}).\n")
            self.control.should_training_stop = True

        scale = self._custom_scale()

        loss_contrast = loss_mle.new_zeros(())
        if use_contrast:
            h_pos = self._pool_hidden(outputs.hidden_states[-1],
                                      model_inputs["labels"])
            loss_contrast = self._contrastive_loss(model, inputs, h_pos)

        loss_rl = loss_mle.new_zeros(())
        reward_val = 0.0
        rl_enabled = (self.gamma != 0.0 or self.gamma_auto) and self.rl_dataset is not None
        if (
            rl_enabled
            and self.state.global_step % self.rl_interval == 0
            and self._last_rl_step != self.state.global_step
        ):
            self._last_rl_step = self.state.global_step
            if self.rl_algo == "grpo":
                loss_rl, reward_val = self._grpo_loss(model)
            else:
                loss_rl, reward_val = self._rl_loss(model)

        if (model.training and self.state.global_step % 10 == 0
                and self._last_log_step != self.state.global_step):
            self._last_log_step = self.state.global_step
            print(f"[Step {self.state.global_step}] CE/token (EMA): {self._ce_ema:.3f} | "
                  f"Contrast[{self.contrast_mode}]: {loss_contrast.item():.3f} | "
                  f"RL: {loss_rl.item():.3f} | Reward: {reward_val:.4f}")

        loss = scale * (self.alpha * loss_mle + self.beta * loss_contrast + loss_rl)
        return (loss, outputs) if return_outputs else loss

print("TwoPhaseTrainerGen defined ✓")


TwoPhaseTrainerGen defined ✓


In [7]:
# ============================================================
# CELL 7: collator + ЧИСТЫЙ eval (фикс бага эха) + коллбэки
#
# БАГ ПРОТОКОЛА D1.x (найден при разборе критики): вход для generate
# СОДЕРЖАЛ золотую метку в конце -> модель продолжала её, и
# «предсказанием» было ЭХО метки. Канарейка D1.5 это показывает
# буквально: «surprised surprised surprised…» — повтор золотой метки.
# Отсюда подозрительно высокие accuracy обученных моделей (0.45–0.65)
# при чистом zero-shot 0.18. Фикс: перед generate отрезаем токены
# метки (labels != -100 считаются от КОНЦА последовательности).
# ============================================================
import json as _json
import torch.nn.functional as F  # ре-импорт: защита от затенения имени F
                                  # (баг: ячейка BERTScore писала P, R, F = ...)


def strip_label_tokens(batch):
    """Отрезает токены метки в конце каждого примера и заново
    лево-пэддит батч. Возвращает (input_ids, attention_mask)
    src-only — модель должна ПРЕДСКАЗАТЬ метку, а не прочитать."""
    labels = batch["labels"]
    n_lab = (labels != -100).sum(dim=1)
    L = batch["input_ids"].size(1)
    srcs = [batch["input_ids"][i, :L - int(n_lab[i])]
            for i in range(batch["input_ids"].size(0))]
    # Маску берём ТЕМ ЖЕ срезом из оригинала: в отрезанном куске
    # остаются исходные лево-паддинговые позиции с маской 0
    # (юнит-тест поймал баг: маска «все единицы» ломала позициями)
    ams = [batch["attention_mask"][i, :L - int(n_lab[i])]
           for i in range(batch["input_ids"].size(0))]
    max_len = max(s.size(0) for s in srcs)
    input_ids = torch.stack([
        s if s.size(0) == max_len
        else F.pad(s, (max_len - s.size(0), 0), value=tok.pad_token_id)
        for s in srcs
    ])
    attention_mask = torch.stack([
        m if m.size(0) == max_len
        else F.pad(m, (max_len - m.size(0), 0), value=0)
        for m in ams
    ])
    return input_ids, attention_mask


@torch.no_grad()
def verbalizer_eval(model, dataset, references, tokenizer, batch_size=16):
    """Вербализатор-скоринг (протокол статьи: top-scoring output token):
    ОДИН forward на src-only входе, argmax по первым токенам 32 меток
    в последней позиции. Детерминированно, без генерации.
    Возвращает (accuracy, correct_vec)."""
    if cfg.task_kind != "emotion_cls":
        return None, None
    model.eval()
    order_labels = sorted(LABEL_FIRST_TOKEN.keys())
    tok_ids = torch.tensor([LABEL_FIRST_TOKEN[l] for l in order_labels],
                           device=model.device)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=causal_lm_collator)
    correct = []
    idx = 0
    for batch in loader:
        input_ids, attention_mask = strip_label_tokens(batch)
        input_ids = input_ids.to(model.device)
        attention_mask = attention_mask.to(model.device)
        logits = model(input_ids=input_ids,
                       attention_mask=attention_mask).logits[:, -1, :]
        pred_idx = logits[:, tok_ids].argmax(dim=-1).tolist()
        for pi in pred_idx:
            correct.append(int(order_labels[pi] == references[idx]))
            idx += 1
    return float(np.mean(correct)), correct


def causal_lm_collator(features):
    pad_token_id = tok.pad_token_id

    def to_1d_tensor(x, dtype=torch.long):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=dtype)
        if x.dim() > 1:
            x = x.squeeze(0)
        return x

    def pad_batch(seqs, value=0):
        max_len = max(s.size(0) for s in seqs)
        return torch.stack([
            s if s.size(0) == max_len
            else F.pad(s, (max_len - s.size(0), 0), value=value)
            for s in seqs
        ])

    # ПАДДИНГ = 0 ДЛЯ ОБОИХ (легаси всех завершённых прогонов).
    # Эмпирически доказано (журнал §6.3): в этом стеке значение пэда
    # input_ids ВЛИЯЕТ на логиты реальных позиций (max diff 6.2 даже
    # при нулевой маске) — «гигиена» pad_token_id НЕ числово-нейтральна,
    # а attention_mask с ненулевыми падами ломает маскирование вовсе.
    # Поэтому: (1) оставляем 0/0 как во всех проведённых экспериментах;
    # (2) замена на pad_token_id возможна ТОЛЬКО при полном перезапуске.
    # Известная историческая особенность: eval-пути (strip_label_tokens,
    # zero-shot через токенизатор) паддят pad_token_id — train/eval
    # рассогласование существовало ВСЕГДА, одинаково для всех конфигов
    # (парные выводы не затронуты), задокументировано в §6.3.
    out = {
        "input_ids": pad_batch([to_1d_tensor(f["input_ids"]) for f in features]),
        "attention_mask": pad_batch(
            [to_1d_tensor(f["attention_mask"]) for f in features]),
    }
    # labels: левый паддинг значением -100
    lab_list = []
    max_len = out["input_ids"].size(1)
    for f in features:
        labs = to_1d_tensor(f["labels"])
        pad_len = max_len - labs.size(0)
        lab_list.append(labs if pad_len == 0 else F.pad(labs, (pad_len, 0), value=-100))
    out["labels"] = torch.stack(lab_list)

    # Негативные последовательности: neg (wrong-label, cls),
    # negoff / negoff2..4 / neginc (D2; D2.3 — динамическое
    # обнаружение любых *_input_ids-колонок негативов)
    neg_pfxs = sorted({k[:-len("_input_ids")] for k in features[0]
                       if k.endswith("_input_ids") and k != "input_ids"})
    for pfx in neg_pfxs:
        if f"{pfx}_attention_mask" in features[0]:
            n_ids = [to_1d_tensor(f[f"{pfx}_input_ids"]) for f in features]
            n_mask = [to_1d_tensor(f[f"{pfx}_attention_mask"]) for f in features]
            n_labs = [to_1d_tensor(f[f"{pfx}_labels"]) for f in features]
            max_len = max(s.size(0) for s in n_ids)
            out[f"{pfx}_input_ids"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=pad_token_id)
                for s in n_ids])
            out[f"{pfx}_attention_mask"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=0)
                for s in n_mask])
            out[f"{pfx}_labels"] = torch.stack([
                s if s.size(0) == max_len
                else F.pad(s, (max_len - s.size(0), 0), value=-100)
                for s in n_labs])

    if "ref_idx" in features[0]:
        out["ref_idx"] = torch.tensor(
            [int(f["ref_idx"]) for f in features], dtype=torch.long
        )

    return out


class Phase1CanaryCallback(TrainerCallback):
    """Каждую эпоху печатает ЧИСТЫЕ генерации (src-only) на val;
    если ВСЕ пустые — останавливает обучение."""

    def __init__(self, canary_ds, tokenizer, collator, first_epoch=2,
                 num_examples=4, max_new_tokens=6):
        n = min(num_examples, len(canary_ds))
        self.loader = DataLoader(
            canary_ds.select(range(n)), batch_size=1, collate_fn=collator
        )
        self.tokenizer = tokenizer
        self.first_epoch = first_epoch
        self.max_new_tokens = max_new_tokens
        self.stopped = False

    @torch.no_grad()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None or state.epoch < self.first_epoch or self.stopped:
            return

        was_training = model.training
        model.eval()
        old_use_cache = getattr(model.config, "use_cache", True)
        model.config.use_cache = True

        texts = []
        try:
            for batch in self.loader:
                input_ids, attention_mask = strip_label_tokens(batch)
                input_ids = input_ids.to(model.device)
                attention_mask = attention_mask.to(model.device)
                out = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                gen = self.tokenizer.decode(
                    out[0, input_ids.shape[1]:], skip_special_tokens=True
                ).strip()
                texts.append(gen if gen else "<EMPTY>")
        finally:
            model.config.use_cache = old_use_cache
            if was_training:
                model.train()

        print(f"[Canary epoch {state.epoch:.0f}] " + " | ".join(t[:80] for t in texts))

        if all(t == "<EMPTY>" for t in texts):
            self.stopped = True
            print("!!! Canary: ВСЕ генерации пустые — модель не обучилась.\n"
                  "    Останавливаю обучение.")
            control.should_training_stop = True


class BestEpochValMetricCallback(TrainerCallback):
    """Раз в эпоху: ЧИСТАЯ метрика (src-only генерация + матчинг)
    на N val-примерах + сохранение ЛУЧШЕГО адаптера в best_prefix.pt."""

    def __init__(self, val_ds, val_references, tokenizer, collator,
                 output_dir, task_kind="emotion_cls",
                 num_examples=200, max_new_tokens=6):
        n = min(num_examples, len(val_ds), len(val_references))
        self.loader = DataLoader(
            val_ds.select(range(n)), batch_size=8, collate_fn=collator
        )
        self.references = list(val_references[:n])
        self.tokenizer = tokenizer
        self.output_dir = output_dir
        self.task_kind = task_kind
        self.metric_name = "accuracy" if task_kind == "emotion_cls" else "rougeL"
        self.max_new_tokens = max_new_tokens
        self.rouge = evaluate.load("rouge") if task_kind != "emotion_cls" else None
        self.best = -1.0
        self.best_epoch = None
        self.history = []
        self._collapse_reported = False

    @torch.no_grad()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None:
            return

        was_training = model.training
        model.eval()
        old_use_cache = getattr(model.config, "use_cache", True)
        model.config.use_cache = True

        preds = []
        try:
            for batch in self.loader:
                input_ids, attention_mask = strip_label_tokens(batch)
                input_ids = input_ids.to(model.device)
                attention_mask = attention_mask.to(model.device)
                out = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                for j in range(input_ids.size(0)):
                    pred = self.tokenizer.decode(
                        out[j, input_ids.shape[1]:], skip_special_tokens=True
                    ).strip()
                    preds.append(pred if pred else " ")
        finally:
            model.config.use_cache = old_use_cache
            if was_training:
                model.train()

        if self.task_kind == "emotion_cls":
            score = float(np.mean([
                match_emotion(p) == r for p, r in zip(preds, self.references)
            ]))
        else:
            score = float(self.rouge.compute(
                predictions=preds, references=self.references,
                rouge_types=["rougeL"],
            )["rougeL"])
        self.history.append({"epoch": round(state.epoch, 2), "value": score})

        if score > self.best:
            self.best = score
            self.best_epoch = round(state.epoch, 2)
            os.makedirs(self.output_dir, exist_ok=True)
            torch.save(
                {n: p.data.detach().cpu().clone()
                 for n, p in model.named_parameters() if p.requires_grad},
                os.path.join(self.output_dir, "best_prefix.pt"),
            )

        # D2: стоп коллапса генерации. Урок ARTICLE s44 (§4.16): CE-плато
        # скрывает деградацию генерации — guard по лоссу её не видит,
        # канарейка видит только ПУСТЫЕ генерации. Здесь: глубокое
        # падение val-метрики относительно лучшей = остановка.
        if (len(self.history) >= 3 and score < 0.5 * self.best
                and not self._collapse_reported):
            self._collapse_reported = True
            print(f"\n!!! Коллапс: val {self.metric_name}={score:.4f} < "
                  f"0.5 x best ({self.best:.4f}) на эпохе {state.epoch:.0f} "
                  f"— останавливаю обучение.\n")
            control.should_training_stop = True
        elif (len(self.history) >= 3 and score < 0.75 * self.best
                and not getattr(self, "_softdeg_warned", False)):
            # критика-2: мягкая деградация (0.5x < val < 0.75x best) —
            # только предупреждение: агрессивный стоп опасен (MLE s44
            # D2.1 восстановился с 0.026 на e1 до 0.176 на e2)
            self._softdeg_warned = True
            print(f"[SoftDegradation] val {self.metric_name}={score:.4f} < "
                  f"0.75 x best ({self.best:.4f}) — предупреждение, "
                  f"БЕЗ остановки")

        with open(os.path.join(self.output_dir, "val_metric_history.json"), "w") as f:
            _json.dump(
                {"metric": self.metric_name, "history": self.history,
                 "best": self.best, "best_epoch": self.best_epoch},
                f, indent=2,
            )

        print(f"[ValMetric epoch {state.epoch:.0f}] {self.metric_name}={score:.4f} | "
              f"best={self.best:.4f} @ epoch {self.best_epoch:.0f}")

    def on_train_end(self, args, state, control, **kwargs):
        print(f"[ValMetric] Итог: best {self.metric_name}={self.best:.4f} @ epoch "
              f"{self.best_epoch:.0f} -> best_prefix.pt")


def prepare_trainer(model, train_ds, val_ds, rl_ds, rl_references, tokenizer, cfg,
                    val_references=None):
    """Конфигурация тренера полностью из cfg — включая output_dir."""

    def compute_metrics(eval_pred):
        return {}

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.total_epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        lr_scheduler_type=cfg.lr_scheduler,
        warmup_steps=cfg.warmup_steps,
        logging_steps=cfg.logging_steps,
        eval_strategy=cfg.eval_strategy,
        save_strategy=cfg.save_strategy,
        save_total_limit=cfg.save_total_limit,
        load_best_model_at_end=cfg.load_best_model_at_end,
        bf16=cfg.bf16,
        fp16=cfg.fp16,
        gradient_checkpointing=cfg.gradient_checkpointing,
        report_to=cfg.report_to,
        remove_unused_columns=cfg.remove_unused_columns,
        dataloader_pin_memory=cfg.dataloader_pin_memory,
        seed=cfg.seed,
        prediction_loss_only=True,
        max_grad_norm=cfg.max_grad_norm,
    )

    canary = Phase1CanaryCallback(
        canary_ds=val_ds,
        tokenizer=tokenizer,
        collator=causal_lm_collator,
        first_epoch=2,
        max_new_tokens=cfg.eval_max_new_tokens,
    )

    callbacks = [canary]

    if val_references is not None:
        callbacks.append(BestEpochValMetricCallback(
            val_ds=val_ds,
            val_references=val_references,
            tokenizer=tokenizer,
            collator=causal_lm_collator,
            output_dir=cfg.output_dir,
            task_kind=cfg.task_kind,
            num_examples=cfg.val_rouge_examples,
            max_new_tokens=cfg.eval_max_new_tokens,
        ))

    trainer = TwoPhaseTrainerGen(
        model=model,
        args=training_args,
        phase1_epochs=cfg.phase1_epochs,
        alpha=cfg.alpha,
        beta=cfg.beta,
        gamma=cfg.gamma,
        gamma_auto=cfg.gamma_auto,
        gamma_target_frac=cfg.gamma_target_frac,
        sigma=cfg.sigma,
        contrast_mode=cfg.contrast_mode,
        contrast_from_start=cfg.contrast_from_start,
        contrastive_dropout=cfg.contrastive_dropout,
        contrastive_tau=cfg.contrastive_tau,
        k_negatives=cfg.k_negatives,
        rl_interval=cfg.rl_interval,
        rl_num_batches=cfg.rl_num_batches,
        rl_subset_size=cfg.rl_subset_size,
        use_reward_baseline=cfg.use_reward_baseline,
        phase2_lr=cfg.phase2_lr,
        divergence_ce_threshold=cfg.divergence_ce_threshold,
        guard_start_step=cfg.guard_start_step,
        rl_dataset=rl_ds,
        rl_references=rl_references,
        tokenizer=tokenizer,
        task_cfg=cfg,
        contrast_neg_in_graph=getattr(cfg, "contrast_neg_in_graph", False),
        reward_baseline_beta=getattr(cfg, "reward_baseline_beta", 0.9),
        rl_algo=getattr(cfg, "rl_algo", "reinforce"),
        grpo_batch=getattr(cfg, "grpo_batch", 2),
        grpo_G=getattr(cfg, "grpo_G", 4),
        grpo_temperature=getattr(cfg, "grpo_temperature", 0.7),
        grpo_top_p=getattr(cfg, "grpo_top_p", 0.9),
        grpo_use_std=getattr(cfg, "grpo_use_std", False),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=causal_lm_collator,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    return trainer

print("collator + strip_label_tokens + verbalizer_eval + коллбэры (чистый eval) ✓")


collator + strip_label_tokens + verbalizer_eval + коллбэры (чистый eval) ✓


In [8]:
def run_experiment(cfg, experiment_name, init_state=None, return_init_state=False):
    """init_state — общий старт пары; return_init_state=True возвращает
    (scores, snapshot). Сохраняет per-example корректности (для
    парного бутстрепа) и вербализатор-скоринг."""

    print(f"\n{'='*70}")
    print(f"STARTING EXPERIMENT: {experiment_name}")
    print(f"{'='*70}\n")

    gc.collect()
    torch.cuda.empty_cache()
    set_seed(cfg.seed)

    dtype = torch.bfloat16 if cfg.bf16 else torch.float32
    base = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        dtype=dtype,
        trust_remote_code=cfg.trust_remote_code,
    )

    if cfg.gradient_checkpointing:
        base.gradient_checkpointing_enable()
    base.config.use_cache = cfg.use_cache

    peft_cfg = PrefixTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        num_virtual_tokens=cfg.num_virtual_tokens,
        prefix_projection=cfg.prefix_projection,
        inference_mode=False,
    )

    model = get_peft_model(base, peft_cfg)
    del base

    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    model.enable_input_require_grads()
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    prompt_param_list = [(n, p) for n, p in model.named_parameters()
                         if p.requires_grad and "prompt_encoder" in n]

    start_snapshot = None
    if init_state is not None:
        with torch.no_grad():
            for n, p in prompt_param_list:
                if n in init_state:
                    p.data.copy_(init_state[n].to(p.device))
        print(f"[PairedStart] загружен ОБЩИЙ старт ({len(init_state)} тензоров "
              f"префикса) — probe пропущен\n")
    elif getattr(cfg, "init_selection_inits", 0) > 0:
        select_best_prefix_init(
            model, train_ds, val_ds, tok, cfg,
            n_inits=cfg.init_selection_inits,
            probe_steps=cfg.init_selection_probe_steps,
        )
        if return_init_state:
            start_snapshot = {n: p.data.detach().cpu().clone()
                              for n, p in prompt_param_list}

    print("Trainable parameters:")
    model.print_trainable_parameters()

    trainer = prepare_trainer(
        model, train_ds, val_ds, rl_ds, rl_references, tok, cfg,
        val_references=val_references,
    )

    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time
    rl_hist = list(getattr(trainer, "rl_reward_history", []) or [])

    print(f"\n{experiment_name} completed in {elapsed/60:.1f} min")

    # Подгружаем ЛУЧШИЙ по val-метрике адаптер
    val_metric_info = {}
    best_path = os.path.join(cfg.output_dir, "best_prefix.pt")
    hist_path = os.path.join(cfg.output_dir, "val_metric_history.json")
    if os.path.exists(best_path):
        best_state = torch.load(best_path, map_location="cpu")
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in best_state:
                    p.data.copy_(best_state[n].to(p.device))
        if os.path.exists(hist_path):
            with open(hist_path) as f:
                val_metric_info = json.load(f)
            print(f"Loaded best adapter: val {val_metric_info['metric']}"
                  f"={val_metric_info['best']:.4f} "
                  f"@ epoch {val_metric_info['best_epoch']:.0f} "
                  f"(история: {[round(h['value'], 4) for h in val_metric_info['history']]})")
        else:
            print("Loaded best adapter (best_prefix.pt)")
    else:
        print("best_prefix.pt не найден — оцениваю последнюю эпоху")

    # ---------- ЧИСТЫЙ тест (src-only generate, без золотой метки) ----------
    print(f"\nEvaluating {experiment_name} on test set (clean protocol)...")
    model.eval()

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        collate_fn=causal_lm_collator,
    )

    predictions = []
    references = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask = strip_label_tokens(batch)
            input_ids = input_ids.to(model.device)
            attention_mask = attention_mask.to(model.device)

            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=cfg.eval_max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )

            for j in range(input_ids.size(0)):
                pred = tok.decode(
                    gen_ids[j, input_ids.shape[1]:], skip_special_tokens=True
                ).strip()
                predictions.append(pred if pred else " ")

    references = list(test_references)

    per_ex = None
    if cfg.task_kind == "emotion_cls":
        pred_labels = [match_emotion(p) for p in predictions]
        correct_vec = [int(pl == r) for pl, r in zip(pred_labels, references)]
        scores = {"accuracy": float(np.mean(correct_vec))}

        # Вербализатор-скоринг (протокол статьи, детерминированный)
        verb_acc, verb_correct = verbalizer_eval(
            model, test_ds, references, tok)
        scores["verb_accuracy"] = verb_acc
    else:
        rouge = evaluate.load("rouge")
        scores = rouge.compute(
            predictions=predictions,
            references=references,
            rouge_types=["rouge1", "rouge2", "rougeL"],
        )
        # D2: per-example ROUGE-L (для парного бутстрепа) + Distinct-1/2
        per_ex = [float(x) for x in rouge.compute(
            predictions=predictions, references=references,
            rouge_types=["rougeL"], use_aggregator=False)["rougeL"]]
        scores["distinct1"], scores["distinct2"] = distinct_n(predictions)
        correct_vec = None
        verb_correct = None
        pred_labels = None

    print(f"\n{'='*70}")
    print(f"RESULTS: {experiment_name}")
    print(f"{'='*70}")
    for k, v in scores.items():
        print(f"{k}: {v:.4f}")
    print(f"{'='*70}\n")

    if pred_labels is not None:
        miss = [(r, pl, p) for r, pl, p in zip(references, pred_labels, predictions)
                if pl != r][:6]
        print("Примеры ошибок (ref -> pred | генерация):")
        for r, pl, p in miss:
            print(f"  {r:<14} -> {str(pl):<14} | {p[:30]!r}")
    else:
        print("Примеры генераций (test, чистый вход):")
        for r, p in list(zip(references, predictions))[:4]:
            print(f"  REF: {r[:90]!r}")
            print(f"  GEN: {p[:90]!r}")

    results = {
        "experiment": experiment_name,
        "config": {
            "task_kind": cfg.task_kind,
            "model": cfg.model_name,
            "beta": cfg.beta,
            "contrast_mode": getattr(cfg, "contrast_mode", None),
            "contrastive_tau": cfg.contrastive_tau,
            "contrast_from_start": getattr(cfg, "contrast_from_start", False),
            "epochs": cfg.total_epochs,
            "lr": cfg.learning_rate,
            "batch": cfg.batch_size,
            "seed": cfg.seed,
            "paired_common_start": init_state is not None,
            "clean_eval": True,
        },
        "test_scores": scores,
        "correct_vec": correct_vec,
        "verb_correct_vec": verb_correct,
        "per_example_rougeL": per_ex,
        "predictions": predictions,
        "rl_reward_history": rl_hist,
        "val_metric": val_metric_info,
        "training_time_min": elapsed / 60,
    }

    output_file = f"{cfg.output_dir}/results.json"
    os.makedirs(cfg.output_dir, exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to {output_file}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    if return_init_state:
        return scores, start_snapshot
    return scores


In [9]:
# ============================================================
# CELL D2-5: конфигурации D2.1 — ядро темы
# ============================================================
class D2_MLE(CFG_D2):
    """Базлайн: чистое CE-дообучение промпта."""
    output_dir = "./gen_d2_mle"
    beta = 0.0


class D2_Mask(D2_MLE):
    """Контроль: маска промпта (рецепт статьи), мягкие значения."""
    output_dir = "./gen_d2_mask"
    beta = 0.1
    contrast_mode = "mask"


class D2_Offtopic(D2_MLE):
    """Ядро темы: негатив = (контекст + ответ ДРУГОГО диалога)."""
    output_dir = "./gen_d2_offtopic"
    beta = 0.1
    contrast_mode = "offtopic"


class D2_Incoherent(D2_MLE):
    """Ядро темы: негатив = (контекст + переставленный золотой ответ)."""
    output_dir = "./gen_d2_incoherent"
    beta = 0.1
    contrast_mode = "incoherent"


cfg2_baseline = D2_MLE()
cfg2_mask = D2_Mask()
cfg2_off = D2_Offtopic()
cfg2_inc = D2_Incoherent()

print("=" * 78)
print("D2.1 CONFIGS (response_gen, Qwen2.5-3B, чистый eval, общий старт)")
print("=" * 78)
for c, name in [(cfg2_baseline, "MLE-only"),
                (cfg2_mask, "contrast-mask (контроль статьи)"),
                (cfg2_off, "contrast-offtopic (ядро)"),
                (cfg2_inc, "contrast-incoherent (ядро)")]:
    print(f"{name:<34} mode={c.contrast_mode:<12} beta={c.beta} -> {c.output_dir}")
print("=" * 78)

D2.1 CONFIGS (response_gen, Qwen2.5-3B, чистый eval, общий старт)
MLE-only                           mode=offtopic     beta=0.0 -> ./gen_d2_mle
contrast-mask (контроль статьи)    mode=mask         beta=0.1 -> ./gen_d2_mask
contrast-offtopic (ядро)           mode=offtopic     beta=0.1 -> ./gen_d2_offtopic
contrast-incoherent (ядро)         mode=incoherent   beta=0.1 -> ./gen_d2_incoherent


In [10]:
# ============================================================
# ЗАПУСК GRPO (C1): 3 сида, пары с MLE D2.1
# ============================================================
import sys
import copy
from math import sqrt

# ЧИСТКА ЗОМБИ-состояния после упавшего прогона: IPython хранит
# traceback последнего исключения (sys.last_traceback), а с ним —
# локали всех кадров, включая модель/оптимизатор (~6 ГБ VRAM).
for _attr in ("last_traceback", "last_value", "last_type"):
    if hasattr(sys, _attr):
        try:
            delattr(sys, _attr)
        except AttributeError:
            pass
gc.collect()
torch.cuda.empty_cache()
_vram = torch.cuda.memory_allocated() / 2**30
print(f"VRAM перед запуском: {_vram:.2f} ГБ занято")
if _vram > 1.0:
    raise RuntimeError(
        f"VRAM не освободилась ({_vram:.2f} ГБ) — перезапусти ядро "
        f"Kernel->Restart и выполни все ячейки заново")
import json


class D2_GRPO(CFG_D2):
    """GRPO-RL: токенный policy gradient с групповым advantage."""
    output_dir = "./gen_d2_grpo"

    beta = 0.0                       # без контраста — чистый тест RL
    contrast_mode = "mask"           # не используется при β=0
    gamma = 1.0                      # стартовое; авто-калибровка перепишет
    gamma_auto = True
    rl_algo = "grpo"
    rl_interval = 100                # 30 событий за прогон
    grpo_batch = 2                   # контекстов на событие
    grpo_G = 4                       # сэмпла на контекст (2×4 = 8 генераций)
    grpo_temperature = 0.7
    grpo_top_p = 0.9
    grpo_use_std = False             # Dr.GRPO: только mean-centering


cfg2_grpo = D2_GRPO()
assert cfg2_grpo.rl_algo == "grpo" and cfg2_grpo.beta == 0.0
print(f"GRPO: событие каждые {cfg2_grpo.rl_interval} шагов, "
      f"{cfg2_grpo.grpo_batch} контекстов × {cfg2_grpo.grpo_G} сэмпла")

_summary_g = f"{cfg2.output_dir}/summary_d2_4.json"
grpo_rows = []
if os.path.exists(_summary_g):
    with open(_summary_g) as f:
        grpo_rows = json.load(f).get("rows", [])

for seed in (42, 43, 44):
    if any(r["seed"] == seed for r in grpo_rows):
        print(f"[Resume] GRPO s{seed} уже завершён")
        continue
    mle_dir = f"{cfg2_baseline.output_dir}_d21_seed{seed}"
    assert os.path.exists(os.path.join(mle_dir, "results.json")), \
        f"нет MLE D2.1 для сида {seed}"
    shared = torch.load(os.path.join(mle_dir, "shared_init.pt"),
                        map_location="cpu")
    with open(os.path.join(mle_dir, "results.json")) as f:
        mle_sc = json.load(f)["test_scores"]
    c = copy.copy(cfg2_grpo)
    c.seed = seed
    c.output_dir = f"{cfg2_grpo.output_dir}_d24_seed{seed}"
    rj = os.path.join(c.output_dir, "results.json")
    if os.path.exists(rj):
        with open(rj) as f:
            sc = json.load(f)["test_scores"]
    else:
        sc = run_experiment(c, f"GRPO D2.4 (seed {seed}, старт D2.1)",
                            init_state=shared)
    grpo_rows = [r for r in grpo_rows if r["seed"] != seed] + [
        {"seed": seed, "MLE-only": mle_sc["rougeL"], "GRPO": sc["rougeL"],
         "distinct1": sc["distinct1"]}]
    del shared
    gc.collect(); torch.cuda.empty_cache()
    with open(_summary_g, "w") as f:
        json.dump({"rows": grpo_rows}, f, indent=2)
    print(f">>> GRPO s{seed}: rougeL={sc['rougeL']:.4f} "
          f"(MLE {mle_sc['rougeL']:.4f}, diff "
          f"{sc['rougeL']-mle_sc['rougeL']:+.4f})\n")

# ---------- агрегация ----------
d = [r["GRPO"] - r["MLE-only"] for r in grpo_rows]
n = len(d)
se = np.std(d, ddof=1) / sqrt(n) if n > 1 else float("nan")
t = np.mean(d) / se if se and se > 0 else float("nan")
print("\n" + "=" * 80)
print(f"GRPO − MLE (генерация, тест 500, n={n} сидов, общий старт)")
print("=" * 80)
print(f"per-seed {[f'{x:+.4f}' for x in d]}")
print(f"mean {np.mean(d):+.4f} ± {np.std(d, ddof=1):.4f} | t={t:+.2f} "
      f"(крит { {2: 12.706, 3: 4.303}.get(n, 3.182) })")
diffs = []
for r in grpo_rows:
    pj = f"./gen_d2_grpo_d24_seed{r['seed']}/results.json"
    mj = f"./gen_d2_mle_d21_seed{r['seed']}/results.json"
    if os.path.exists(pj) and os.path.exists(mj):
        va = json.load(open(pj)).get("per_example_rougeL")
        vb = json.load(open(mj)).get("per_example_rougeL")
        if va and vb:
            diffs += [x - y for x, y in zip(va, vb)]
if diffs:
    arr = np.array(diffs, float)
    rng = np.random.default_rng(42)
    boots = np.array([arr[rng.integers(0, len(arr), len(arr))].mean()
                      for _ in range(10000)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    print(f"бутстреп: {arr.mean():+.4f} [CI {lo:+.4f}, {hi:+.4f}] p={p:.4f}")

print("\nЗдоровье GRPO по прогонам (награда/разброс групп):")
for r in grpo_rows:
    pj = f"./gen_d2_grpo_d24_seed{r['seed']}/results.json"
    if os.path.exists(pj):
        h = json.load(open(pj)).get("rl_reward_history") or []
        if h:
            print(f"  s{r['seed']}: событий {len(h)} | reward "
                  f"{np.mean([x['reward'] for x in h[:5]]):.4f} -> "
                  f"{np.mean([x['reward'] for x in h[-5:]]):.4f} | "
                  f"group_std {np.mean([x.get('group_std', float('nan')) for x in h]):.4f}")

print("\nКонтекст: параметрический REINFORCE (D2.2, n=7) = +0.0005 (t=0.30);")
print("RL-sample (D2.2b) = +0.0007. Вопрос C1: даёт ли токенный")
print("group-relative градиент то, чего не дали обе формы выше?")
with open(_summary_g, "w") as f:
    json.dump({"rows": grpo_rows}, f, indent=2)
print(f"\nИтог: {_summary_g}")

VRAM перед запуском: 0.00 ГБ занято
GRPO: событие каждые 100 шагов, 2 контекстов × 4 сэмпла

STARTING EXPERIMENT: GRPO D2.4 (seed 42, старт D2.1)



Loading weights: 100%|██████████| 434/434 [00:00<00:00, 1305.26it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[GRPO] калибровка коэффициента отложена
[GRPO step 0] reward=0.1268 group_std=0.0613 coef=1.0000
[GRPO sample] REF: 'Was this a friend you were in love with, or just a best friend?'
[GRPO sample] GEN: ' Sounds like a great day. Do you still have the pictures?' (r=0.160) | GEN: ' That sounds like a great time. I wish I had a best friend like that.' (r=0.276)
[Step 0] GRPO-шаг (CE замещён) | RL: -0.010 | Reward: 0.1268


Epoch,Training Loss,Validation Loss
1,2.269205,2.317945
2,2.386276,2.308135
3,2.303488,2.307269
4,2.159944,2.320088


[Step 10] CE/token (EMA): 2.435 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.336 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.296 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.338 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.273 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.285 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.252 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.264 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.255 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[GRPO] коэффициент откалиброван: 1.0000
[GRPO step 100] reward=0.1699 group_std=0.0246 coef=1.0000
[GRPO sample] REF: 'That is good, maybe you can get a raise.'
[GRPO sample] GEN: " That's awesome! Are you going to take it?" (r=0.222) 

...

[Step 10] CE/token (EMA): 2.251 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.254 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.266 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.243 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.266 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.308 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.248 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.189 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.243 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[GRPO] коэффициент откалиброван: 1.0000
[GRPO step 100] reward=0.1506 group_std=0.0554 coef=1.0000
[GRPO sample] REF: 'That is good, maybe you can get a raise.'
[GRPO sample] GEN: ' That is good news! Are you excited about the new job 

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 9861.47it/s]


[PairedStart] загружен ОБЩИЙ старт (5 тензоров префикса) — probe пропущен

Trainable parameters:
trainable params: 4,807,936 || all params: 3,090,746,624 || trainable%: 0.1556
[Trainer] Prefix params collected: 5
[Trainer] Contrast: mode=mask, from_start=True, beta=0.0
[GRPO] калибровка коэффициента отложена
[GRPO step 0] reward=0.0499 group_std=0.0483 coef=1.0000
[GRPO sample] REF: 'Was this a friend you were in love with, or just a best friend?'
[GRPO sample] GEN: ' That sounds fun!' (r=0.000) | GEN: ' I love fireworks. Did you get to see the whole show?' (r=0.080)
[Step 0] GRPO-шаг (CE замещён) | RL: 0.003 | Reward: 0.0499


Epoch,Training Loss,Validation Loss
1,2.358218,2.303919
2,2.234765,2.314423
3,2.146434,2.310705
4,2.163628,2.318175


[Step 10] CE/token (EMA): 2.299 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 20] CE/token (EMA): 2.269 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 30] CE/token (EMA): 2.237 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 40] CE/token (EMA): 2.245 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 50] CE/token (EMA): 2.199 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 60] CE/token (EMA): 2.159 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 70] CE/token (EMA): 2.235 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 80] CE/token (EMA): 2.213 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[Step 90] CE/token (EMA): 2.281 | Contrast[mask]: 0.000 | RL: 0.000 | Reward: 0.0000
[GRPO] коэффициент откалиброван: 1.0000
[GRPO step 100] reward=0.1698 group_std=0.0354 coef=1.0000
[GRPO sample] REF: 'That is good, maybe you can get a raise.'
[GRPO sample] GEN: " That's great to hear! What kind of job are you gettin